<a href="https://colab.research.google.com/github/Cheth-code/jugaad-ml/blob/main/diffusers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Core HF & diffusion libs
!pip install --upgrade diffusers[torch] transformers accelerate safetensors huggingface_hub xformers

# PyTorch + CUDA (11.8)
!pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

# Imaging
!pip install Pillow

In [ ]:
from diffusers import StableDiffusionPipeline
import torch
from IPython.display import display, clear_output
from PIL import Image

pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16,
).to("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def show_gaussian_steps(pipeline, step_index, timestep, callback_kwargs):
    # extract the latents tensor
    latents = callback_kwargs["latents"]
    # undo VAE scaling
    lat = latents * (1 / pipeline.vae.config.scaling_factor)
    # decode to image tensor and normalize
    img_tensor = pipeline.vae.decode(lat).sample[0]
    img = (img_tensor / 2 + 0.5).clamp(0, 1)
    # convert and display
    pil_img = pipeline.numpy_to_pil(img.cpu().permute(1, 2, 0).numpy())[0]
    display(pil_img)
    return callback_kwargs

# 3. Run with inline previews
torch.manual_seed(42)
prompt = "universe imagenation"

_ = pipe(
    prompt,
    num_inference_steps=20,                           # 3 noisy previews + final pass
    guidance_scale=7.5,
    callback_on_step_end=show_gaussian_steps,
    callback_on_step_end_tensor_inputs=["latents"],  # ensure latents is passed into callback
)